<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/gemma_chat"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td>    <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/gemma_chat.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/gemma_chat.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fgemma_chat.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/gemma_chat.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

# 使用 Gemma 建造 chatbot

Gemma 等大型語言模型 (LLM) 擅長產生資訊豐富的回應，這使它們成為構建虛擬助理和 chatbot 的理想選擇。
傳統上，法學碩士以無狀態的方式運作，這意味著他們缺乏儲存過去對話的固有記憶。每個 prompt 或問題都是獨立處理的，忽略先前的互動。然而，自然對話的一個重要方面是保留先前互動中的上下文的能力。為了克服這一限制並使法學碩士能夠維護對話上下文，必須明確向他們提供相關信息，例如向法學碩士提交的每個新prompt中的對話歷史記錄（或相關部分）。
本教學向您展示如何使用Gemma 的指令調整模型變體來開發chatbot。

## 設定

### Gemma設置

要完成本教學，您首先需要完成 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明。 Gemma 設定說明向您展示如何執行以下操作：
* 在 kaggle.com 上造訪Gemma。
* Select a Colab runtime with sufficient resources to run
Gemma 2B 型號。* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝依賴項

Install Keras and KerasNLP.

In [ ]:
# Install Keras 3 last. See https://keras.io/getting_started/ for more details.
!pip install -q tensorflow-cpu
!pip install -q -U keras-nlp tensorflow-hub
!pip install -q -U "keras>=3"
!pip install -q -U tensorflow-text

### 選擇後端

Keras 是一個高級、多framework 深度學習API，設計簡單易用。 [Keras 3](https://keras.io/keras_3){:.external} 可讓您選擇後端：TensorFlow、JAX 或 PyTorch。這三個都適用於本教學。

In [ ]:
import os

# Select JAX as the backend
os.environ["KERAS_BACKEND"] = "jax"

# Pre-allocate 100% of TPU memory to minimize memory fragmentation
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.0"

### 導入包

導入 Keras 和 KerasNLP。

In [ ]:
import keras
import keras_nlp

# for reproducibility
keras.utils.set_random_seed(42)

### 實例化模型

KerasNLP 提供了許多流行的[模型架構](https://keras.io/api/keras_nlp/models/){:.external} 的實作。在本教學中，您將使用 `GemmaCausalLM` 實例化模型，這是用於因果語言建模的端對端 Gemma 模型。因果語言模型根據前一個 tokens 預測下一個 token。
使用 `from_preset` 方法實例化模型：

In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_instruct_2b_en")

`GemmaCausalLM.from_preset()` 函數根據預設的架構和權重實例化模型。在上面的程式碼中，字串`"gemma2_instruct_2b_en"`指定了具有20億個參數的預設Gemma 2 2B 模型。另提供帶有[7B、9B 和27B 參數](https://ai.google.com/gemma/docs/get_started#models-list) 的Gemma 型號。您可以在 [Kaggle](https://www.kaggle.com/models/google/gemma) 上的 **型號變體** 清單中找到 Gemma 型號的程式碼字串。
注意：要執行 Colab 中的較大模型，您需要存取付費方案中提供的高級 GPU。或者，您可以使用Kaggle notebooks 或Google Cloud 項目執行inferences。

使用`summary`方法來取得更多關於模型的資訊：

In [ ]:
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                   │                                             256,000 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

從摘要中可以看到，該模型有 26 億個可訓練參數。
注意：為了命名模型（“2B”），嵌入層不計入參數數量。

### Define formatting helper functions

In [ ]:
from IPython.display import Markdown
import textwrap

def display_chat(prompt, text):
  formatted_prompt = "<font size='+1' color='brown'>🙋‍♂️<blockquote>" + prompt + "</blockquote></font>"
  text = text.replace('•', '  *')
  text = textwrap.indent(text, '> ', predicate=lambda _: True)
  formatted_text = "<font size='+1' color='teal'>🤖\n\n" + text + "\n</font>"
  return Markdown(formatted_prompt+formatted_text)

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

## 建造chatbot

Gemma指令調整模型`gemma2_instruct_2b_en`微調以理解以下轉tokens：
```
<start_of_turn>user\n  ... <end_of_turn>\n
<start_of_turn>model\n ... <end_of_turn>\n
```

本教學使用這些tokens 來建構chatbot。有關Gemma 控制tokens 的詳細信息，請參閱[格式化和系統指令](https://ai.google.dev/gemma/docs/formatting)。

### 建立聊天助理來管理對話狀態

In [ ]:
class ChatState():
  """
  Manages the conversation history for a turn-based chatbot
  Follows the turn-based conversation guidelines for the Gemma family of models
  documented at https://ai.google.dev/gemma/docs/formatting
  """

  __START_TURN_USER__ = "<start_of_turn>user\n"
  __START_TURN_MODEL__ = "<start_of_turn>model\n"
  __END_TURN__ = "<end_of_turn>\n"

  def __init__(self, model, system=""):
    """
    Initializes the chat state.

    Args:
        model: The language model to use for generating responses.
        system: (Optional) System instructions or bot description.
    """
    self.model = model
    self.system = system
    self.history = []

  def add_to_history_as_user(self, message):
      """
      Adds a user message to the history with start/end turn markers.
      """
      self.history.append(self.__START_TURN_USER__ + message + self.__END_TURN__)

  def add_to_history_as_model(self, message):
      """
      Adds a model response to the history with start/end turn markers.
      """
      self.history.append(self.__START_TURN_MODEL__ + message)

  def get_history(self):
      """
      Returns the entire chat history as a single string.
      """
      return "".join([*self.history])

  def get_full_prompt(self):
    """
    Builds the prompt for the language model, including history and system description.
    """
    prompt = self.get_history() + self.__START_TURN_MODEL__
    if len(self.system)>0:
      prompt = self.system + "\n" + prompt
    return prompt

  def send_message(self, message):
    """
    Handles sending a user message and getting a model response.

    Args:
        message: The user's message.

    Returns:
        The model's response.
    """
    self.add_to_history_as_user(message)
    prompt = self.get_full_prompt()
    response = self.model.generate(prompt, max_length=2048)
    result = response.replace(prompt, "")  # Extract only the new response
    self.add_to_history_as_model(result)
    return result


### 與模特兒聊天

開始與模特兒聊天。

In [ ]:
chat = ChatState(gemma_lm)
message = "Tell me, in a few words,  how to compute all prime numbers up to 1000?"
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Tell me, in a few words,  how to compute all prime numbers up to 1000?</blockquote></font><font size='+1' color='teal'>🤖

> **Sieve of Eratosthenes.** 
> <end_of_turn>
</font>

繼續談話。

In [ ]:
message = "Now in Python! No numpy, please!"
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Now in Python! No numpy, please!</blockquote></font><font size='+1' color='teal'>🤖

> ```python
> def sieve_of_eratosthenes(n):
>   """Returns a list of prime numbers up to n."""
>   primes = [True] * (n + 1)
>   primes[0] = primes[1] = False
>   for i in range(2, int(n**0.5) + 1):
>     if primes[i]:
>       for j in range(i * i, n + 1, i):
>         primes[j] = False
>   return [i for i, is_prime in enumerate(primes) if is_prime]
> 
> primes = sieve_of_eratosthenes(1000)
> print(primes)
> ```
> 
> **Explanation:**
> 
> 1. **Initialization:**
>    - `primes = [True] * (n + 1)`: Creates a list `primes` of boolean values, initially assuming all numbers are prime.
>    - `primes[0] = primes[1] = False`: Sets 0 and 1 as non-prime.
> 
> 2. **Iteration:**
>    - `for i in range(2, int(n**0.5) + 1):`: Iterates from 2 to the square root of `n`. We only need to check up to the square root because any composite number must have a prime factor less than or equal to its square root.
>    - `if primes[i]:`: If `i` is marked as prime:
>      - `for j in range(i * i, n + 1, i):`: Marks all multiples of `i` as non-prime.
> 
> 3. **Result:**
>    - `return [i for i, is_prime in enumerate(primes) if is_prime]`: Creates a list of indices where `primes[i]` is True, representing the prime numbers.
> 
> 
> Let me know if you'd like a more detailed explanation of any part! 
> <end_of_turn>
</font>

In [ ]:
message = "Thank you, it works! Can you explain the code in French?"
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Thank you, it works! Can you explain the code in French?</blockquote></font><font size='+1' color='teal'>🤖

> Bien sûr ! Voici une explication du code en français :
> 
> ```python
> def sieve_of_eratosthenes(n):
>   """Retourne une liste de nombres premiers jusqu'à n."""
>   primes = [True] * (n + 1)
>   primes[0] = primes[1] = False
>   for i in range(2, int(n**0.5) + 1):
>     if primes[i]:
>       for j in range(i * i, n + 1, i):
>         primes[j] = False
>   return [i for i, is_prime in enumerate(primes) if is_prime]
> 
> primes = sieve_of_eratosthenes(1000)
> print(primes)
> ```
> 
> **Explication:**
> 
> 1. **Initialisation:**
>    - `primes = [True] * (n + 1)`: Crée une liste `primes` de valeurs booléennes, initialement supposant que tous les nombres sont premiers.
>    - `primes[0] = primes[1] = False`: Définit 0 et 1 comme non-premiers.
> 
> 2. **Itération:**
>    - `for i in range(2, int(n**0.5) + 1):`: Itère de 2 jusqu'à la racine carrée de `n`. Nous ne devons vérifier que jusqu'à la racine carrée car tout nombre composite doit avoir un facteur premier inférieur ou égal à sa racine carrée.
>    - `if primes[i]:`: Si `i` est considéré comme premier:
>      - `for j in range(i * i, n + 1, i):`:  Marquer tous les multiples de `i` comme non-premiers.
> 
> 3. **Resultat:**
>    - `return [i for i, is_prime in enumerate(primes) if is_prime]`: Crée une liste des indices où `primes[i]` est vrai, représentant les nombres premiers.
> 
> 
> N'hésitez pas à me demander si vous avez besoin d'une explication plus détaillée de quelque chose! 
> <end_of_turn>
</font>

In [ ]:
message = "Great! Now add those explanations as comments in the code."
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Great! Now add those explanations as comments in the code.</blockquote></font><font size='+1' color='teal'>🤖

> ```python
> def sieve_of_eratosthenes(n):
>   """Retourne une liste de nombres premiers jusqu'à n."""
>   # Initialise une liste de boolean avec True pour tous les nombres de 0 à n
>   primes = [True] * (n + 1)
>   # Définit 0 et 1 comme non-premiers
>   primes[0] = primes[1] = False
>   # Itère de 2 à la racine carrée de n
>   for i in range(2, int(n**0.5) + 1):
>     # Si i est considéré comme premier
>     if primes[i]:
>       # Itère sur tous les multiples de i
>       for j in range(i * i, n + 1, i):
>         # Définit les multiples de i comme non-premiers
>         primes[j] = False
>   # Retourne la liste des indices des nombres premiers
>   return [i for i, is_prime in enumerate(primes) if is_prime]
> 
> primes = sieve_of_eratosthenes(1000)
> print(primes)
> ```
> 
> **Explication:**
> 
> * **Initialisation:**
>     * `primes = [True] * (n + 1)`:  Crée une liste `primes` de valeurs booléennes, initialement supposant que tous les nombres sont premiers.
>     * `primes[0] = primes[1] = False`: Définit 0 et 1 comme non-premiers.
> * **Itération:**
>     * `for i in range(2, int(n**0.5) + 1):`: Itère de 2 jusqu'à la racine carrée de `n`. Nous ne devons vérifier que jusqu'à la racine carrée car tout nombre composite doit avoir un facteur premier inférieur ou égal à sa racine carrée.
>     * `if primes[i]:`: Si `i` est considéré comme premier:
>         * `for j in range(i * i, n + 1, i):`:  Marquer tous les multiples de `i` comme non-premiers.
> * **Resultat:**
>     * `return [i for i, is_prime in enumerate(primes) if is_prime]`: Crée une liste des indices où `primes[i]` est vrai, représentant les nombres premiers. 
> 
> 
> 
> <end_of_turn>
</font>

透過執行生成的程式碼來測試產生的回應：

In [ ]:
def sieve_of_eratosthenes(n):
  """Retourne une liste de nombres premiers jusqu'à n."""
  # Initialise une liste de boolean avec True pour tous les nombres de 0 à n
  primes = [True] * (n + 1)
  # Définit 0 et 1 comme non-premiers
  primes[0] = primes[1] = False
  # Itère de 2 à la racine carrée de n
  for i in range(2, int(n**0.5) + 1):
    # Si i est considéré comme premier
    if primes[i]:
      # Itère sur tous les multiples de i
      for j in range(i * i, n + 1, i):
        # Définit les multiples de i comme non-premiers
        primes[j] = False
  # Retourne la liste des indices des nombres premiers
  return [i for i, is_prime in enumerate(primes) if is_prime]

primes = sieve_of_eratosthenes(1000)
print(primes)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503, 509, 521, 523, 541, 547, 557, 563, 569, 571, 577, 587, 593, 599, 601, 607, 613, 617, 619, 631, 641, 643, 647, 653, 659, 661, 673, 677, 683, 691, 701, 709, 719, 727, 733, 739, 743, 751, 757, 761, 769, 773, 787, 797, 809, 811, 821, 823, 827, 829, 839, 853, 857, 859, 863, 877, 881, 883, 887, 907, 911, 919, 929, 937, 941, 947, 953, 967, 971, 977, 983, 991, 997]


使用`get_history` 方法檢視`Chat` 類別如何保留所有上下文。

In [ ]:
print(chat.get_history())

<start_of_turn>user
Tell me, in a few words,  how to compute all prime numbers up to 1000?<end_of_turn>
<start_of_turn>model
**Sieve of Eratosthenes.** 
<end_of_turn><start_of_turn>user
Now in Python! No numpy, please!<end_of_turn>
<start_of_turn>model
```python
def sieve_of_eratosthenes(n):
  """Returns a list of prime numbers up to n."""
  primes = [True] * (n + 1)
  primes[0] = primes[1] = False
  for i in range(2, int(n**0.5) + 1):
    if primes[i]:
      for j in range(i * i, n + 1, i):
        primes[j] = False
  return [i for i, is_prime in enumerate(primes) if is_prime]

primes = sieve_of_eratosthenes(1000)
print(primes)
```

**Explanation:**

1. **Initialization:**
   - `primes = [True] * (n + 1)`: Creates a list `primes` of boolean values, initially assuming all numbers are prime.
   - `primes[0] = primes[1] = False`: Sets 0 and 1 as non-prime.

2. **Iteration:**
   - `for i in range(2, int(n**0.5) + 1):`: Iterates from 2 to the square root of `n`. We only need to check up to

## 總結和進一步閱讀

在本教學中，您學習如何在JAX 上使用Keras 與Gemma 2B 指令調整模型聊天。
查看這些指南和教學以了解有關 Gemma 的更多資訊：
* [開始使用Keras Gemma](https://ai.google.dev/gemma/docs/get_started)。
* [在 GPU 上微調 Gemma 型號](https://ai.google.dev/gemma/docs/lora_tuning)。
* 了解 [Gemma 與 Vertex AI 整合](https://ai.google.dev/gemma/docs/integrations/vertex)
* 了解如何[將 Gemma 模型與 Vertex AI 一起使用](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma){:.external}。
